In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/MP-PCA
!ls

/content/drive/MyDrive/MP-PCA
add_noise.ipynb  localpca_dn.py     noisy_data	 Real_Imag_Combine.ipynb
denoised	 meas_gre_dir1.mat  __pycache__  시각화.ipynb


In [3]:
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import os

In [4]:
mat = sio.loadmat("meas_gre_dir1.mat")

meas_gre = mat["meas_gre"]

print(meas_gre.shape)
print(meas_gre.dtype)
print(np.iscomplexobj(meas_gre))

(256, 224, 176, 6)
complex128
True


In [5]:
noise_mat = sio.loadmat(
    "noisy_data/noisy_meas_gre_dir1_10.mat"
)

print(noise_mat.keys())

dict_keys(['__header__', '__version__', '__globals__', 'noisy_real', 'noisy_imag', 'sigma'])


In [6]:
noisy_real = noise_mat["noisy_real"].astype(np.float32)
noisy_imag = noise_mat["noisy_imag"].astype(np.float32)

print("real :", noisy_real.shape)
print("imag :", noisy_imag.shape)

real : (256, 224, 176, 6)
imag : (256, 224, 176, 6)


In [7]:
combined_input = np.concatenate(
    [noisy_real, noisy_imag],
    axis=-1
)

print("Combined shape :", combined_input.shape)
print("dtype :", combined_input.dtype)
print("complex? :", np.iscomplexobj(combined_input))

Combined shape : (256, 224, 176, 12)
dtype : float32
complex? : False


In [8]:
print(mat.keys())

dict_keys(['__header__', '__version__', '__globals__', 'B0_dir', 'CF', 'delta_TE', 'mask_brain', 'meas_gre', 'te_gre', 'tolvalue', 'voxel_size_gre'])


In [9]:
!pip install dipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.7/184.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.9/397.9 kB 21.2 MB/s eta 0:00:00


In [10]:
import importlib.util

file_path = "/content/drive/MyDrive/MP-PCA/localpca_dn.py"

spec = importlib.util.spec_from_file_location(
    "localpca_dn",
    file_path
)

localpca_dn = importlib.util.module_from_spec(spec)
spec.loader.exec_module(localpca_dn)

print("로드 성공")
print(localpca_dn.__file__)

로드 성공
/content/drive/MyDrive/MP-PCA/localpca_dn.py


In [11]:
from dipy.denoise.localpca import mppca as dipy_mppca
mppca_noclip = localpca_dn.mppca
import numpy as np

# Echo 0에서 값이 0이 아닌 영역을 임시 brain mask로 사용
mask_brain = np.abs(meas_gre[..., 0]) > 0

print(mask_brain.shape)
print(mask_brain.dtype)

(256, 224, 176)
bool


In [12]:
import os
import numpy as np
import scipy.io as sio

os.makedirs("denoised", exist_ok=True)

In [13]:
import os
import numpy as np
import scipy.io as sio

# 저장 폴더
os.makedirs("denoised", exist_ok=True)

# 처리할 noise level
noise_levels = [10, 30, 50]

for level in noise_levels:

    print(f"\n===== {level}% 시작 =====")

    # noisy data 불러오기
    noise_mat = sio.loadmat(
        f"noisy_data/noisy_meas_gre_dir1_{level}.mat"
    )

    noisy_real = noise_mat["noisy_real"].astype(np.float32)
    noisy_imag = noise_mat["noisy_imag"].astype(np.float32)

    # Real + Imag 결합
    combined_input = np.concatenate(
        [noisy_real, noisy_imag],
        axis=-1
    )

    print("combined shape:", combined_input.shape)

    # -------------------------
    # 1. 기본 DIPY MP-PCA
    # -------------------------
    den_dipy = dipy_mppca(
        combined_input,
        mask=mask_brain,
        patch_radius=3
    )

    sio.savemat(
        f"denoised/combined_dipy_{level}.mat",
        {
            "combined_denoised": den_dipy
        }
    )

    print(f"{level}% DIPY 저장 완료")

    # -------------------------
    # 2. clipping 제거 MP-PCA
    # -------------------------
    den_noclip = mppca_noclip(
        combined_input,
        mask=mask_brain,
        patch_radius=3
    )

    sio.savemat(
        f"denoised/combined_noclip_{level}.mat",
        {
            "combined_denoised": den_noclip
        }
    )

    print(f"{level}% no-clip 저장 완료")

    print(f"===== {level}% 완료 =====")


===== 10% 시작 =====
combined shape: (256, 224, 176, 12)


/usr/local/lib/python3.13/dist-packages/dipy/denoise/localpca.py:377: RuntimeWarning: invalid value encountered in divide
  denoised_arr = thetax / theta


10% DIPY 저장 완료


/content/drive/MyDrive/MP-PCA/localpca_dn.py:376: RuntimeWarning: invalid value encountered in divide
  denoised_arr = thetax / theta


10% no-clip 저장 완료
===== 10% 완료 =====

===== 30% 시작 =====
combined shape: (256, 224, 176, 12)
30% DIPY 저장 완료
30% no-clip 저장 완료
===== 30% 완료 =====

===== 50% 시작 =====
combined shape: (256, 224, 176, 12)
50% DIPY 저장 완료
50% no-clip 저장 완료
===== 50% 완료 =====
